In [ ]:
!pip install -U transformers datasets accelerate scikit-learn pandas matplotlib seaborn torch

In [ ]:
pip show pyarrow

In [ ]:
import sys
print(sys.executable)

import pyarrow
print(pyarrow.__file__)
print(getattr(pyarrow, "__version__", "NO VERSION"))

In [ ]:
pip install pandas pyarrow

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from torch import nn
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    set_seed
)

from datasets import Dataset

In [ ]:
try:
    BASE_DIR = os.path.dirname(
        os.path.dirname(os.path.abspath(__file__))
    )
except NameError:
    BASE_DIR = os.getcwd()

PROCESSED = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

OUTPUT = os.path.join(
    BASE_DIR,
    "outputs",
    "security_classifier"
)

MODEL_OUTPUT = os.path.join(
    OUTPUT,
    "best_model"
)

TOKENIZER_OUTPUT = os.path.join(
    OUTPUT,
    "tokenizer"
)

os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(MODEL_OUTPUT, exist_ok=True)
os.makedirs(TOKENIZER_OUTPUT, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Processed:", PROCESSED)
print("Output:", OUTPUT)

In [ ]:
SEED = 42

set_seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation")

PROCESSED = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_csv(
    os.path.join(
        PROCESSED,
        "security_train_processed.csv"
    )
)

validation_df = pd.read_csv(
    os.path.join(
        PROCESSED,
        "security_validation_processed.csv"
    )
)

test_df = pd.read_csv(
    os.path.join(
        PROCESSED,
        "security_test_processed.csv"
    )
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

In [ ]:
print(train_df.columns.tolist())

In [ ]:
print("Train missing prompts:",
      train_df["prompt"].isna().sum())

print("Validation missing prompts:",
      validation_df["prompt"].isna().sum())

print("Test missing prompts:",
      test_df["prompt"].isna().sum())

In [ ]:
print("Training Distribution")
print(train_df["attack_type"].value_counts())

print("\nValidation Distribution")
print(validation_df["attack_type"].value_counts())

print("\nTest Distribution")
print(test_df["attack_type"].value_counts())

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation")

label_mapping = pd.read_csv(
    BASE_DIR / "outputs" / "tokenized" / "label_mapping.csv"
)

label_mapping

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation")

label_mapping = pd.read_csv(
    BASE_DIR / "outputs" / "tokenized" / "label_mapping.csv"
)

# Explicit mapping — DO NOT use LabelEncoder
label2id = dict(
    zip(
        label_mapping["attack_type"],
        label_mapping["label"]
    )
)

id2label = {
    int(v): k
    for k, v in label2id.items()
}

print("label2id:")
print(label2id)

print("\nid2label:")
print(id2label)

In [ ]:
print(
    "Unique training labels:",
    sorted(train_df["label"].unique())
)

print(
    "Expected labels:",
    sorted(id2label.keys())
)

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded:", MODEL_NAME)

In [ ]:
token_lengths = train_df["prompt"].apply(
    lambda x: len(
        tokenizer(
            str(x),
            truncation=False,
            add_special_tokens=True
        )["input_ids"]
    )
)

token_lengths.describe()

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    token_lengths,
    bins=40
)

plt.xlabel("Number of Tokens")
plt.ylabel("Number of Prompts")
plt.title("Training Prompt Token Length Distribution")

plt.tight_layout()

plt.show()

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df[
        ["prompt", "label"]
    ],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df[
        ["prompt", "label"]
    ],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[
        ["prompt", "label"]
    ],
    preserve_index=False
)

print(train_dataset)

In [ ]:
MAX_LENGTH = 256
def tokenize_function(examples):

    return tokenizer(
        examples["prompt"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["prompt"]
)

validation_dataset = validation_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["prompt"]
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["prompt"]
)

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
)

num_classes = len(label2id)

total_samples = len(train_df)

class_weights = total_samples / (
    num_classes * class_counts
)

class_weights = class_weights.sort_index()

print("Class counts:")
print(class_counts)

print("\nClass weights:")
print(class_weights)

In [ ]:
class_weights_tensor = torch.tensor(
    class_weights.values,
    dtype=torch.float
)

class_weights_tensor

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
)

print("Model loaded.")

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    precision_weighted, recall_weighted, f1_weighted, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }

In [ ]:
class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):

        labels = inputs.get("labels")

        outputs = model(
            **inputs
        )

        logits = outputs.get("logits")

        weights = class_weights_tensor.to(
            logits.device
        )

        loss_function = nn.CrossEntropyLoss(
            weight=weights
        )

        loss = loss_function(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (
            loss,
            outputs
        ) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(

    output_dir=OUTPUT,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=32,

    num_train_epochs=4,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",

    greater_is_better=True,

    save_total_limit=2,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=SEED
)

In [ ]:
trainer = WeightedTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()

In [ ]:
validation_results = trainer.evaluate(
    validation_dataset
)

validation_results

In [ ]:
trainer.save_model(
    MODEL_OUTPUT
)

tokenizer.save_pretrained(
    TOKENIZER_OUTPUT
)

print("Best model saved.")

In [ ]:
trainer.save_model(
    MODEL_OUTPUT
)

tokenizer.save_pretrained(
    TOKENIZER_OUTPUT
)

print("Best model saved.")

In [ ]:
history = pd.DataFrame(
    trainer.state.log_history
)

history

In [ ]:
history.to_csv(
    os.path.join(
        OUTPUT,
        "training_history.csv"
    ),
    index=False
)

print("Training history saved.")

In [ ]:
loss_history = history[
    history["loss"].notna()
]

plt.figure(figsize=(9, 5))

plt.plot(
    loss_history["epoch"],
    loss_history["loss"],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss")

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT,
        "training_loss.png"
    ),
    dpi=300
)

plt.show()

In [ ]:
loss_history = history[
    history["loss"].notna()
]

plt.figure(figsize=(9, 5))

plt.plot(
    loss_history["epoch"],
    loss_history["loss"],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss")

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT,
        "training_loss.png"
    ),
    dpi=300
)

plt.show()

In [ ]:
# Get evaluation history from Hugging Face Trainer
eval_history = [
    log for log in trainer.state.log_history
    if "eval_accuracy" in log
]

print(eval_history)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    [log["epoch"] for log in eval_history],
    [log["eval_accuracy"] for log in eval_history],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT,
        "validation_accuracy.png"
    ),
    dpi=300
)

plt.show()

In [ ]:
best_f1 = max(
    [log["eval_f1_macro"] for log in eval_history]
)

best_accuracy = max(
    [log["eval_accuracy"] for log in eval_history]
)

summary = f"""
========================================
SECURITY CLASSIFIER TRAINING SUMMARY
========================================

Model:
{MODEL_NAME}

Maximum Sequence Length:
{MAX_LENGTH}

Training Samples:
{len(train_df)}

Validation Samples:
{len(validation_df)}

Test Samples:
{len(test_df)}

Number of Classes:
{num_classes}

Epochs:
{training_args.num_train_epochs}

Learning Rate:
{training_args.learning_rate}

Batch Size:
{training_args.per_device_train_batch_size}

Best Validation Macro F1:
{best_f1:.4f}

Best Validation Accuracy:
{best_accuracy:.4f}

Classes:
{id2label}

========================================
"""

print(summary)

with open(
    os.path.join(
        OUTPUT,
        "training_summary.txt"
    ),
    "w",
    encoding="utf-8"
) as f:

    f.write(summary)

In [ ]:
print("Training completed.")

print("\nOutput files:")

for root, dirs, files in os.walk(OUTPUT):

    for file in files:

        print(
            os.path.join(
                root,
                file
            )
        )